## Triggered LFP


For this tutorial we use a dataset from chronic recordings collected for Steinmetz et al. 2021. This is a subset of the data available [here](https://rdr.ucl.ac.uk/articles/dataset/Chronic_recordings_from_Neuropixels_2_0_probes_in_mice/24411841).

The dataset was collected by _Anna Lebedeva_ in the Carandini/Harris lab at UCL from animal _AL032_, and prepared by _Jennifer Colonell_ (HHMI, Janelia). 

**Only data from shank0 and recording day 2019-11-21 are provided.**

For experimental details about the recording (specifically: the experiment in Figure 4 of the paper, Supplementary methods 1.2 Chronic recordings in Carandini/Harris Lab at UCL)

Read more about [Current Source Density](https://www.neuroinf.pl/Members/danek/homepage/preprints/Article.2013-12-16.4021/getFile)


In [ ]:
from pathlib import Path
import numpy as np 
import pylab as plt
from helpers import *
# for interactivity
%matplotlib ipympl 

stimfolder = Path('D:/course_data/AL032_stimulus')
foldername = Path('D:/course_data/AL032/AL032/AL032_2019-11-21_stripe192-natIm_g0/AL032_2019-11-21_stripe192-natIm_g0_imec1000')  # path to folder containing the binary file
# try to find the binary file (has extension .bin)
fname = list(foldername.rglob('*.bin'))
if not len(fname): # display an error message if the file is not found.
    raise(ValueError(f'Could not find the file in folder {foldername}'))
else:
    fname = fname[0] # the full path to the file

# load the binary file
dat,meta = load_spikeglx_binary(fname)

# get the units conversion factors
digitizer_range_volts = meta['imAiRangeMax'] # of niAiRangeMax for NI streams or obAiRangeMax for onebox streams
digitizer_max_value = meta['imMaxInt'] # or niMaxInt for NI streams or obMaxInt for onebox stream

# the gain is fixed for NP2 but variable for 1.0 probes
if int(meta['imDatPrb_type']) in [21,24]:
    gain = 80
elif int(meta['imDatPrb_type']) in [2013]:
    gain = 100

conversion_factor = digitizer_range_volts/digitizer_max_value/gain * 1e6 # convert to microvolt my multiplying by 1e6

sampling_rate = meta['imSampRate'] # from the .meta

from scipy.io import loadmat # load matlab files in python
mat = loadmat(stimfolder.joinpath('AL032_stimulus_times.mat'),squeeze_me=True)
# get the stim times
stimtimes = mat['sorted_response'][np.where(mat['day_label'] == '2019-11-21')].squeeze()
stimtimes = np.stack(stimtimes) # ntrial x nstim x feature


Get triggered LFP, we will extract the average first. 


In [ ]:
def lfp_filter(X,freq_low = 3., freq_high = 150, original_sampling_rate = 30000, downsampling_rate = 5000):
    # define a function to perform the filtering (isolate the LF band and downsample)
    from scipy.signal import butter, filtfilt, decimate
    Wn = np.array([freq_low,freq_high]) / (original_sampling_rate/2.) # define the filter band
    b, a = butter(3, Wn, btype = 'band') # third order butterworth filter
    nX = filtfilt(b,a,X, axis = 0) # use 2 passes
    return decimate(nX,int(original_sampling_rate/downsampling_rate),axis = 0)

tpre = -0.3
tpost = 0.3
idx = np.arange(tpre*sampling_rate, tpost*sampling_rate, dtype = int)

time = np.arange(len(dat))/sampling_rate


#get the channelmap from the snsGeomMap 
cmap = np.stack([[int(a) for a in s.split(':')] for s in meta['snsGeomMap'][1:]])
cmap = cmap[cmap[:,0]==0,:][:,[1,2]] # shank zero
# sort the depth and get every other channel because we are taking a single column
channelidx = np.argsort(cmap[:,1])
cmap = cmap[channelidx]
channelidx = channelidx[cmap[:,0]==np.min(cmap[:,0])]
cmap = cmap[cmap[:,0]==np.min(cmap[:,0]),:]


# get all trials
trial_lfp = []
for t_stim in tqdm(stimtimes[:,:,1].flatten()):
    t_sample = np.where(time<=t_stim)[0][-1]
    chunk = dat[t_sample+idx,:-1][:,channelidx] # drop the sync channel
    trial_lfp.append(lfp_filter(chunk))
trial_lfp = np.stack(trial_lfp)

In [ ]:
# average LFP
avg_lfp = trial_lfp.mean(axis = 0).T

t = np.linspace(tpre,tpost,avg_lfp.
shape[1])
plt.figure()

plt.imshow(avg_lfp*conversion_factor,aspect = 'auto',
           extent = [tpre,tpost,cmap[0,1],cmap[-1,1]],
           clim = [-200,200], cmap = 'RdBu_r')
plt.colorbar(shrink = 0.3,label = '$\mu$V')

for p,depth in zip(avg_lfp[::1],cmap[::1,1]):
     plt.plot(t,p/10+depth,'k',lw = 0.5,clip_on = False)
plt.ylim([cmap[0,1],cmap[-1,1]])
# plt.xticks([])
plt.ylabel('Channel position ($\mu$m)',fontsize = 14)
plt.xlabel('Time from stim onset (s)',fontsize = 14);


The current source density is the second spatial derivative of the LFP.

The supplementary methods of _Laminar Structure of Spontaneous and Sensory-Evoked Population Activity in Auditory Cortex_ has a nice description on how to compute it [Sakata and Harris 2009](https://www.sciencedirect.com/science/article/pii/S089662730900720X)

In [ ]:
# CSD
def current_source_density(lfp,chmap, interpolation_chanspacing=30):
    # Interpolate so that we get even sampling
    ux = np.unique(chmap[:,0])
    ix = np.argmax([np.sum(chmap[:,0]==u) for u in ux])
    chidx = chmap[:,0]==ux[ix]
    y = np.array(chmap[chidx,1])
    duration = lfp.shape[1]
    x = np.arange(duration)
    z = lfp[:,:]
    from scipy.interpolate import RegularGridInterpolator
    f = RegularGridInterpolator((x, y), z.T,bounds_error=False,method='nearest')
    # f = interp2d(x,y,z)
    ny = np.arange(np.min(y)-interpolation_chanspacing,np.max(y)+interpolation_chanspacing,interpolation_chanspacing)
    xxnew, yynew = np.meshgrid(x,ny, indexing='ij')
    nlfp = f((xxnew,yynew)).T
    # duplicate outmost channels
    csd = np.empty((nlfp.shape[0]-2,nlfp.shape[1]))
    smoothed_lfp = np.empty_like(nlfp)
    for i in range(csd.shape[0]):
        smoothed_lfp[i+1,:] = (1./4.) *(nlfp[i,:] + 2.*nlfp[i+1,:] + nlfp[i+2,:])
    smoothed_lfp[0,:] = (1./4.) *(3.*nlfp[0,:] + nlfp[1,:])
    smoothed_lfp[-1,:] = (1./4.) *(3.*nlfp[-1,:] + nlfp[-2,:])
    for i in range(csd.shape[0]):
        csd[i,:] = -(1./(interpolation_chanspacing*1.e-3)**2.)*(smoothed_lfp[i,:]-2.*smoothed_lfp[i+1,:]+smoothed_lfp[i+2,:])
    
    a,b,z = (x,np.linspace(np.min(y)-interpolation_chanspacing,np.max(y)+interpolation_chanspacing,csd.shape[0]),csd)
    f = RegularGridInterpolator((a, b), z.T,bounds_error=False)
    ny = np.arange(np.min(y)-interpolation_chanspacing,np.max(y)+interpolation_chanspacing,5.)
    xxnew, yynew = np.meshgrid(x,ny, indexing='ij')

    return f((xxnew,yynew)).T,smoothed_lfp[:,1:-1]


t = np.linspace(tpre,tpost,avg_lfp.shape[0])
plt.figure()
csd,smoothlfp = current_source_density(avg_lfp*conversion_factor,cmap)
plt.imshow(csd,aspect = 'auto',
        extent = [tpre,tpost,cmap[0,1],cmap[-1,1]], cmap = 'jet',clim = [-8000,8000])
cb = plt.colorbar(shrink = 0.3,label = 'csd')

plt.ylim([cmap[0,1],cmap[-1,1]])
plt.ylabel('Channel position ($\mu$m)',fontsize = 14)
plt.xlabel('Time from stim onset (s)',fontsize = 14);
plt.xlim([-0.0,0.07])
plt.xticks([0])
plt.ylim([3000,3500]);
